# Objetivo del Script
Limpiar y preparar un conjunto de datos de propiedades para su posterior análisis. Esto incluye la estandarización de valores, la conversión de tipos de datos y la extracción de información geográfica clave.

## Resumen del flujo
1.  Cargar y concatenar múltiples archivos CSV de propiedades.
2.  Normalizar la columna de precios, convirtiendo a una moneda única (MXN) y a formato numérico.
3.  Limpiar y convertir a tipos numéricos columnas como `Estacionamientos`, `Baños`, `Recamaras` y `Superficie`.
4.  Limpiar y convertir a tipo numérico la columna `Mantenimiento`.
5.  Extraer información de `Colonia` y `Municipio` de la columna `Direccion`.
6.  Crear una columna `Estado` con un valor predefinido.

---

# Archivos de Entrada
-   Múltiples archivos CSV.

# Archivos de Salida
-  Aarchivo CSV intermedio con los datos combinados y renombrada la columna de precio inicial.
-   Un DataFrame pandas (`combined_df`) limpio y preprocesado en memoria para análisis posteriores.

---

# Librerías Utilizadas
-   **pandas** → manipulación y análisis de datos.
-   **glob** → búsqueda de archivos por patrón.
-   **numpy** → operaciones numéricas, especialmente para valores nulos.
-   **re** → expresiones regulares para la limpieza de texto.

---

# 1. Carga y Concatenación de Datos
**Acciones**
-   Se identifican todos los archivos CSV que coinciden con el patrón.
-   Se lee cada archivo CSV y se renombra la primera columna a `Precio`.
-   Todos los DataFrames individuales se concatenan en un único DataFrame llamado `combined_df`.
-   El DataFrame combinado se guarda como y luego se vuelve a cargar para asegurar la persistencia.

**Puntos clave**
-   El proceso asegura que todos los datos de las hojas separadas se unifiquen para un análisis consistente.

---

# 2. Normalización de Precios (Moneda)
**Acciones**
-   Se identifica la moneda (`MN` o `USD`) en la columna `Precio`.
-   Los valores en USD se convierten a MXN utilizando un tipo de cambio fijo (`1 USD = 19.89 MXN`).
-   Se eliminan los prefijos de moneda (`MN`, `USD`), comas y espacios extra.
-   La columna resultante, `Precio_MXN`, se convierte a tipo `float64`.
-   La columna `Precio` original es eliminada.

**Puntos clave**
-   Se estandarizan todos los precios a Pesos Mexicanos (MXN) para permitir comparaciones directas.

---

# 3. Limpieza y Conversión de Columnas Numéricas
**Acciones**
-   **Estacionamientos:** Se elimina el texto `'estac.'` y se convierte la columna a tipo entero (`Int64`) que admite valores nulos.
-   **Baños:** Se eliminan los textos `'baños'` y `'baño'` y se convierte la columna a tipo entero (`Int64`) que admite valores nulos.
-   **Recamaras:** Se manejan los rangos de recámaras (ej. `'3 a 5 rec.'`), calculando el promedio redondeado hacia abajo. Se elimina el texto `'rec.'` y se convierte la columna a tipo entero (`Int64`) que admite valores nulos.
-   **Superficie:** Se manejan los rangos de superficie (ej. `'100 a 200 m² lote'`), calculando el promedio. Se elimina el texto `'m² lote'` y se convierte la columna a tipo flotante (`float64`) que admite valores nulos.

**Puntos clave**
-   Se preparan estas columnas para análisis numéricos, asegurando que los valores faltantes se manejen adecuadamente con tipos que soportan `NaN`.

---

# 4. Limpieza y Conversión de `Mantenimiento`
**Acciones**
-   Se eliminan los prefijos `MN`, el texto `Mantenimiento` y las comas de la columna.
-   La columna se convierte a tipo entero (`Int64`) que admite valores nulos.

**Puntos clave**
-   Estandariza los costos de mantenimiento para un análisis numérico.

---

# 5. Creación de Columnas Geográficas
**Acciones**
-   Se divide la columna `Direccion` para extraer la `Colonia` (primer elemento) y el `Municipio` (segundo elemento).
-   Se crea una nueva columna `Estado` y se asigna el valor `'Estado de México'` a todas las filas.

**Puntos clave**
-   Permite segmentar y analizar las propiedades por su ubicación geográfica a un nivel más granular.

---

In [ ]:
import pandas as pd
import glob

# Buscar todos los archivos CSV que coinciden con el patrón 'inmuebles24*.csv'
csv_files = glob.glob('/content/inmuebles24*.csv')

df_list = []

# Iterar sobre cada archivo CSV encontrado
for csv_file in csv_files:
    # Leer el archivo CSV en un DataFrame
    df = pd.read_csv(csv_file)
    # Renombrar la primera columna (que contiene el precio) a 'Precio'
    df.rename(columns={df.columns[0]: 'Precio'}, inplace=True)
    # Añadir el DataFrame a la lista
    df_list.append(df)

# Concatenar todos los DataFrames en uno solo, ignorando los índices originales
combined_df = pd.concat(df_list, ignore_index=True)

display(combined_df.head())
display(combined_df.info())
display(combined_df.describe())

# Guardar el DataFrame combinado en un nuevo archivo CSV
combined_df.to_csv('inmuebles24_galo.csv', index=False)

In [ ]:
# Contar el número de filas en la columna 'Precio' que contienen 'MN' (Pesos Mexicanos)
count_mn = combined_df[combined_df['Precio'].str.contains('MN', na=False)].shape[0]
print(f"Número de entradas con 'MN': {count_mn}")

# Contar el número de filas en la columna 'Precio' que contienen 'USD' (Dólares Estadounidenses)
count_usd = combined_df[combined_df['Precio'].str.contains('USD', na=False)].shape[0]
print(f"Número de entradas con 'USD': {count_usd}")

# Mostrar las filas donde 'Precio' no contiene 'MN' (útil para identificar otros formatos o USD)
display(combined_df[~combined_df['Precio'].str.contains('MN', na=False)])

2979
21


In [ ]:
# Inicializar 'Precio_MXN' copiando la columna original 'Precio' para facilitar el procesamiento.
combined_df['Precio_MXN'] = combined_df['Precio']

# Identificar las filas que contienen 'USD' y realizar la conversión.
# Primero, limpia el texto, convierte a numérico y aplica el tipo de cambio.
usd_mask = combined_df['Precio_MXN'].str.contains('USD', na=False)
combined_df.loc[usd_mask, 'Precio_MXN'] = combined_df.loc[usd_mask, 'Precio_MXN'].str.replace('USD', '', regex=False).str.replace(',', '', regex=False).str.strip().astype(float) * 19.89

# Para las filas restantes (presumiblemente en MN o sin moneda específica), limpia el texto.
# Elimina 'MN', comas y espacios, luego convierte a tipo float.
# Aseguramos que la columna sea de tipo string antes de aplicar str.replace
combined_df['Precio_MXN'] = combined_df['Precio_MXN'].astype(str).str.replace('MN', '', regex=False).str.replace(',', '', regex=False).str.strip().astype(float)

# Eliminar la columna original 'Precio' ya que ahora tenemos 'Precio_MXN' estandarizado.
combined_df = combined_df.drop(columns=['Precio'])

# Eliminar el texto 'estac.' de la columna 'Estacionamientos' para prepararla para la conversión numérica.
combined_df['Estacionamientos'] = combined_df['Estacionamientos'].str.replace('estac.', '', regex=False)

# Mostrar la suma de valores nulos en 'Estacionamientos' antes de la conversión final.
display(combined_df[['Estacionamientos']].isna().sum())

# Convertir la columna 'Estacionamientos' a un tipo entero que admite valores nulos (Int64).
combined_df['Estacionamientos'] = combined_df['Estacionamientos'].astype('Int64')

# Mostrar las primeras 50 filas de 'Baños' para inspección.
display(combined_df[['Baños']].head(50))
# Mostrar las últimas 50 filas de 'Baños' para inspección.
display(combined_df[['Baños']].tail(50))
# Mostrar la suma de valores nulos en 'Baños' para inspección.
display(combined_df[['Baños']].isna().sum())

In [ ]:
# Eliminar el texto 'baños' de las entradas en la columna 'Baños'.
combined_df['Baños'] = combined_df['Baños'].str.replace('baños', '', regex=False)
# Eliminar el texto 'baño' de las entradas en la columna 'Baños'.
combined_df['Baños'] = combined_df['Baños'].str.replace('baño', '', regex=False)
# Convertir la columna 'Baños' a un tipo entero que admite valores nulos (Int64).
combined_df['Baños'] = combined_df['Baños'].astype('Int64')

# Mostrar las primeras 50 filas de 'Recamaras' para inspección.
display(combined_df[['Recamaras']].head(50))
# Mostrar las últimas 50 filas de 'Recamaras' para inspección.
display(combined_df[['Recamaras']].tail(50))
# Mostrar la suma de valores nulos en 'Recamaras' para inspección.
display(combined_df[['Recamaras']].isna().sum())

In [ ]:
import numpy as np
import re

# Función para manejar rangos como 'm a n' en la columna 'Recamaras'.
def handle_recamaras_range(rec):
    if isinstance(rec, str):
        # Buscar el patrón 'm a n' usando expresiones regulares.
        match = re.search(r'(\d+)\s*a\s*(\d+)', rec)
        if match:
            m = int(match.group(1))
            n = int(match.group(2))
            # Calcular el promedio y redondear hacia abajo.
            return str(int(np.floor((m + n) / 2)))
        else:
            # Si no es un rango, eliminar 'rec.' y espacios extra.
            return rec.replace('rec.', '', regex=False).strip()
    return rec # Devolver valores no string (como NaN) tal cual están.

# Aplicar la función a la columna 'Recamaras' para limpiar y estandarizar los valores.
combined_df['Recamaras'] = combined_df['Recamaras'].apply(handle_recamaras_range)

# Convertir la columna 'Recamaras' a un tipo entero que admite valores nulos (Int64).
combined_df['Recamaras'] = combined_df['Recamaras'].astype('Int64')

In [ ]:
display(combined_df[['Recamaras']].head(50))
display(combined_df[['Recamaras']].tail(50))
display(combined_df[['Recamaras']].isna().sum())
display(combined_df[['Superficie']].head(50))
display(combined_df[['Superficie']].tail(50))
display(combined_df[['Superficie']].isna().sum())

,Superficie
0,339 m² lote
1,238 m² lote
2,230 m² lote
3,256 m² lote
4,162 m² lote
5,105 m² lote
6,300 m² lote
7,154 m² lote
8,2300 m² lote
9,237 m² lote


,Superficie
2950,249 m² lote
2951,431 m² lote
2952,56 m² lote
2953,85 m² lote
2954,429 m² lote
2955,400 m² lote
2956,170 m² lote
2957,327 m² lote
2958,200 m² lote
2959,1200 m² lote


,0
Superficie,7


In [ ]:
import numpy as np
import re

# Función para manejar rangos como 'm a n' en la columna 'Superficie'.
def handle_superficie_range(superficie_val):
    if isinstance(superficie_val, str):
        # Buscar el patrón 'm a n' usando expresiones regulares.
        match = re.search(r'(\d+)\s*a\s*(\d+)', superficie_val)
        if match:
            m = int(match.group(1))
            n = int(match.group(2))
            # Calcular el promedio.
            return str(((m + n) / 2))
        else:
            # Si no es un rango, eliminar 'm² lote' y espacios extra.
            return superficie_val.replace('m² lote', '', regex=False).strip()
    return superficie_val # Devolver valores no string (como NaN) tal cual están.

# Aplicar la función a la columna 'Superficie' para limpiar y estandarizar los valores.
combined_df['Superficie'] = combined_df['Superficie'].apply(handle_superficie_range)

# Convertir la columna 'Superficie' a un tipo flotante que admite valores nulos (float64).
combined_df['Superficie'] = combined_df['Superficie'].astype('float64')

# Mostrar las primeras 50 filas de 'Superficie' para inspección.
display(combined_df[['Superficie']].head(50))
# Mostrar las últimas 50 filas de 'Superficie' para inspección.
display(combined_df[['Superficie']].tail(50))
# Mostrar la suma de valores nulos en 'Superficie' para inspección.
display(combined_df[['Superficie']].isna().sum())

# Mostrar las primeras 50 filas de 'Direccion' para inspección.
display(combined_df[['Direccion']].head(50))
# Mostrar las últimas 50 filas de 'Direccion' para inspección.
display(combined_df[['Direccion']].tail(50))
# Mostrar la suma de valores nulos en 'Direccion' para inspección.
display(combined_df[['Direccion']].isna().sum())

# Crear una nueva columna "Colonia" extrayendo el texto antes de ", " en la columna "Direccion".
combined_df['Colonia'] = combined_df['Direccion'].str.split(', ').str[0]
# Crear una nueva columna "Municipio" extrayendo el texto después de ", " en la columna "Direccion".
combined_df['Municipio'] = combined_df['Direccion'].str.split(', ').str[1]

# Crear una nueva columna "Estado" y asignar el valor 'Estado de México' a todas las filas.
combined_df['Estado'] = 'Estado de México'

# Mostrar las primeras 50 filas de 'Mantenimiento' para inspección.
display(combined_df[['Mantenimiento']].head(50))
# Mostrar las últimas 50 filas de 'Mantenimiento' para inspección.
display(combined_df[['Mantenimiento']].tail(50))
# Mostrar la suma de valores nulos en 'Mantenimiento' para inspección.
display(combined_df[['Mantenimiento']].isna().sum())

In [ ]:
import numpy as np

def clean_mantenimiento(value):
    # Si el valor es de tipo string, limpiar el texto.
    if isinstance(value, str):
        # Eliminar 'MN', ' Mantenimiento' y comas, luego eliminar espacios extra.
        return value.replace('MN', '', regex=False).replace(' Mantenimiento', '', regex=False).replace(',', '', regex=False).strip()
    # Devolver NaN para valores no string (incluyendo NaNs originales).
    return np.nan

# Aplicar la función de limpieza a la columna 'Mantenimiento'.
combined_df['Mantenimiento'] = combined_df['Mantenimiento'].apply(clean_mantenimiento)

# Convertir la columna a numérica, forzando errores a NaN, y luego a un tipo entero que admite nulos (Int64).
combined_df['Mantenimiento'] = pd.to_numeric(combined_df['Mantenimiento'], errors='coerce').astype('Int64')

# Mostrar las primeras 50 filas de 'Mantenimiento' después de la limpieza.
display(combined_df[['Mantenimiento']].head(50))
# Mostrar las últimas 50 filas de 'Mantenimiento' después de la limpieza.
display(combined_df[['Mantenimiento']].tail(50))
# Mostrar la suma de valores nulos en 'Mantenimiento' después de la limpieza.
display(combined_df[['Mantenimiento']].isna().sum())